## Phol Castañeda Henao, Neithan Gómez Rivera y Daniela Higuita Agudelo

##### Entrega 4

In [3]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

ROOT_DIR = Path().resolve().parent
DATA_DIR = ROOT_DIR / "data/raw"
file_path = 'nyc-rolling-sales.csv'



df = pd.read_csv(DATA_DIR / file_path,
    na_values=[' ',' -  ']

)
df

,Unnamed: 0,BOROUGH,NEIGHBORHOOD,BUILDING CLASS CATEGORY,TAX CLASS AT PRESENT,BLOCK,LOT,EASE-MENT,BUILDING CLASS AT PRESENT,ADDRESS,...,RESIDENTIAL UNITS,COMMERCIAL UNITS,TOTAL UNITS,LAND SQUARE FEET,GROSS SQUARE FEET,YEAR BUILT,TAX CLASS AT TIME OF SALE,BUILDING CLASS AT TIME OF SALE,SALE PRICE,SALE DATE
0,4,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,392,6,NaN,C2,153 AVENUE B,...,5,0,5,1633.0,6440.0,1900,2,C2,6625000.0,2017-07-19 00:00:00
1,5,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,26,NaN,C7,234 EAST 4TH STREET,...,28,3,31,4616.0,18690.0,1900,2,C7,NaN,2016-12-14 00:00:00
2,6,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,399,39,NaN,C7,197 EAST 3RD STREET,...,16,1,17,2212.0,7803.0,1900,2,C7,NaN,2016-12-09 00:00:00
3,7,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,402,21,NaN,C4,154 EAST 7TH STREET,...,10,0,10,2272.0,6794.0,1913,2,C4,3936272.0,2016-09-23 00:00:00
4,8,1,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2A,404,55,NaN,C2,301 EAST 10TH STREET,...,6,0,6,2369.0,4615.0,1900,2,C2,8000000.0,2016-11-17 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84543,8409,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7349,34,NaN,B9,37 QUAIL LANE,...,2,0,2,2400.0,2575.0,1998,1,B9,450000.0,2016-11-28 00:00:00
84544,8410,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7349,78,NaN,B9,32 PHEASANT LANE,...,2,0,2,2498.0,2377.0,1998,1,B9,550000.0,2017-04-21 00:00:00
84545,8411,5,WOODROW,02 TWO FAMILY DWELLINGS,1,7351,60,NaN,B2,49 PITNEY AVENUE,...,2,0,2,4000.0,1496.0,1925,1,B2,460000.0,2017-07-05 00:00:00
84546,8412,5,WOODROW,22 STORE BUILDINGS,4,7100,28,NaN,K6,2730 ARTHUR KILL ROAD,...,0,7,7,208033.0,64117.0,2001,4,K6,11693337.0,2016-12-21 00:00:00


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84548 entries, 0 to 84547
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Unnamed: 0                      84548 non-null  int64  
 1   BOROUGH                         84548 non-null  int64  
 2   NEIGHBORHOOD                    84548 non-null  object 
 3   BUILDING CLASS CATEGORY         84548 non-null  object 
 4   TAX CLASS AT PRESENT            83810 non-null  object 
 5   BLOCK                           84548 non-null  int64  
 6   LOT                             84548 non-null  int64  
 7   EASE-MENT                       0 non-null      float64
 8   BUILDING CLASS AT PRESENT       83810 non-null  object 
 9   ADDRESS                         84548 non-null  object 
 10  APARTMENT NUMBER                19052 non-null  object 
 11  ZIP CODE                        84548 non-null  int64  
 12  RESIDENTIAL UNITS               

In [5]:
#Descartando variables
df = df.drop(columns=['Unnamed: 0','EASE-MENT','APARTMENT NUMBER','TOTAL UNITS','BLOCK','LOT'])
# Eliminando registros duplicados despues de eliminar variables que no aportan información
df =  df.drop_duplicates()

df = df.dropna(subset=['SALE PRICE','TAX CLASS AT PRESENT','BUILDING CLASS AT PRESENT'])

# Eliminando valores de 0 de SALE PRICE
df = df[df['SALE PRICE'] > 1000]

# Eliminando valores de 0 de YEAR BUILT ya que no aportan información
df = df[df['YEAR BUILT'] > 0]

df['SALE DATE'] = pd.to_datetime(df['SALE DATE'])

df['SALE YEAR'] = df['SALE DATE'].dt.year
df['SALE MONTH'] = df['SALE DATE'].dt.month
df['SALE DAY'] = df['SALE DATE'].dt.day

df = df.drop(columns=['SALE DATE'])


#Tratando outliers se encontró que al usar el metodo del IQR
# se perdían demasiados datos, por ello se decidió dejar el resultado del
# tratamiento con percentiles.

df_percentiles = df.copy()

numerical_cols = [
    'RESIDENTIAL UNITS',
    'COMMERCIAL UNITS',
    'LAND SQUARE FEET',
    'GROSS SQUARE FEET',
    'YEAR BUILT',
    'SALE YEAR',
    'SALE MONTH',
    'SALE DAY',
    'SALE PRICE'
]

for col in numerical_cols:
    lower = df_percentiles[col].quantile(0.01)
    upper = df_percentiles[col].quantile(0.99)

    df_percentiles = df_percentiles[
        (df_percentiles[col] >= lower) &
        (df_percentiles[col] <= upper)
    ]
df_percentiles = df_percentiles.drop(columns=['ADDRESS'])

print('Número de registros:', df_percentiles.shape[0])


Número de registros: 32877


In [6]:
df.isna().sum()


BOROUGH                               0
NEIGHBORHOOD                          0
BUILDING CLASS CATEGORY               0
TAX CLASS AT PRESENT                  0
BUILDING CLASS AT PRESENT             0
ADDRESS                               0
ZIP CODE                              0
RESIDENTIAL UNITS                     0
COMMERCIAL UNITS                      0
LAND SQUARE FEET                  18932
GROSS SQUARE FEET                 19089
YEAR BUILT                            0
TAX CLASS AT TIME OF SALE             0
BUILDING CLASS AT TIME OF SALE        0
SALE PRICE                            0
SALE YEAR                             0
SALE MONTH                            0
SALE DAY                              0
dtype: int64

Mismas particiones para todos los modelos:

In [7]:
from sklearn.model_selection import train_test_split

X = df_percentiles.drop(['SALE PRICE'], axis=1)
y = df_percentiles['SALE PRICE']

#Partiendo el dataset en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, train_size=0.8)
print(f'Tamaño del conjunto de entrenamiento es: {X_train.shape}')
print(f'Tamaño del conjunto de prueba es: {X_test.shape}')

Tamaño del conjunto de entrenamiento es: (26301, 16)
Tamaño del conjunto de prueba es: (6576, 16)


## MODELO LINEAL (RIDGECV)

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, PolynomialFeatures, StandardScaler, PowerTransformer, TargetEncoder

ss = StandardScaler()
pt = PowerTransformer()

imputer = SimpleImputer(strategy='median')

proc_num_ss = Pipeline([
    ('imputer', imputer), #imputando valores faltantes con la mediana (para el modelo lineal) desde el pipeline con simpleimputer
    ('ss', ss)
    ])


ordenTaxPresent = ['1', '1A', '1B', '1C', '2', '2A', '2B', '2C', '3', '4']
ordenTaxTimeOfSale = ['1','2','3','4']

oreTaxPresent = OrdinalEncoder(categories=[ordenTaxPresent], dtype='int')
oreTaxTimeOfSale = OrdinalEncoder(categories=[ordenTaxTimeOfSale], dtype='int')
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore' )



te = TargetEncoder(target_type='continuous', random_state=1)



In [9]:
from sklearn.compose import ColumnTransformer

preprocessorLineal = ColumnTransformer(transformers=[
    ('num_prep', proc_num_ss, ['LAND SQUARE FEET', 'GROSS SQUARE FEET']), # Escalado
    ('num_prep2', pt, ['RESIDENTIAL UNITS','COMMERCIAL UNITS','YEAR BUILT', 'SALE YEAR', 'SALE MONTH', 'SALE DAY']), # Transformación
    ('cod_taxPresent', oreTaxPresent, ['TAX CLASS AT PRESENT']), # Codificación ordinal
    ('cod_taxTimeOfSale', oreTaxTimeOfSale, ['TAX CLASS AT TIME OF SALE']), # Codificación ordinal
    ('cod_oh', ohe, ['BOROUGH']),
    ('cod_teCardinalidad', te, ['NEIGHBORHOOD', 'BUILDING CLASS AT PRESENT', 'ZIP CODE', 'BUILDING CLASS AT TIME OF SALE','BUILDING CLASS CATEGORY'])
    ], remainder='passthrough') # Las demás columnas se descartan



In [10]:
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso

from sklearn.linear_model import RidgeCV
from sklearn.metrics import root_mean_squared_error
import warnings
warnings.filterwarnings("ignore")

alphas = np.logspace(-4, 4)


model = Pipeline([

    # Preprocesamiento
    ('preprocessor', preprocessorLineal),

    #Corrección: "La selección de características debería estar dentro del pipeline."

    # Selección de características con Lasso en el pipeline
    ('feature_selection', SelectFromModel( Lasso(alpha=0.1, random_state=1))),

    # Modelo final
    ('regressor', RidgeCV(alphas=alphas, cv=5,scoring='neg_root_mean_squared_error'))

])

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# MÉTRICAS
ridge_model = model.named_steps['regressor']

print('Métricas')
print('Best RMSE:', -ridge_model.best_score_)
print('Best alpha:', ridge_model.alpha_)

print(
    'Train RMSE:',
    root_mean_squared_error(y_train, train_pred)
)

print(
    'Test RMSE:',
    root_mean_squared_error(y_test, test_pred)
)


Métricas
Best RMSE: 421057.6324350076
Best alpha: 75.43120063354607
Train RMSE: 407635.9155557427
Test RMSE: 420376.4567594613


In [11]:
features = model.named_steps[
    'preprocessor'
].get_feature_names_out()


selected_mask = model.named_steps[
    'feature_selection'
].get_support()

selected_features = features[selected_mask]


# COEFICIENTES DEL RIDGE
coefs = model.named_steps[
    'regressor'
].coef_

# DATAFRAME FINAL

coef_df_ridge = pd.DataFrame({
    'feature': selected_features,
    'coef': coefs
})

coef_df_ridge = coef_df_ridge.sort_values(
    by='coef',
    key=abs,
    ascending=False
)

coef_df_ridge

,feature,coef
1,num_prep__GROSS SQUARE FEET,187016.909486
2,num_prep2__RESIDENTIAL UNITS,-111886.955206
0,num_prep__LAND SQUARE FEET,73574.486638
11,cod_oh__BOROUGH_4,-69418.150444
12,cod_oh__BOROUGH_5,-62186.702208
8,cod_taxTimeOfSale__TAX CLASS AT TIME OF SALE,-51307.653085
10,cod_oh__BOROUGH_3,37180.480811
4,num_prep2__SALE YEAR,29552.601326
7,cod_taxPresent__TAX CLASS AT PRESENT,-17374.178011
5,num_prep2__SALE MONTH,16466.865867


## Arbol de Decisión

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder

#Preprocessor sin numéricos ya que los árboles y boosting no lo requieren

preprocessor = ColumnTransformer(transformers=[
    ('cod_taxPresent', oreTaxPresent, ['TAX CLASS AT PRESENT']), # Codificación ordinal
    ('cod_taxTimeOfSale', oreTaxTimeOfSale, ['TAX CLASS AT TIME OF SALE']), # Codificación ordinal
    ('cod_oh', ohe, ['BOROUGH']),
    ('cod_teCardinalidad', te, ['NEIGHBORHOOD', 'BUILDING CLASS AT PRESENT', 'ZIP CODE', 'BUILDING CLASS AT TIME OF SALE','BUILDING CLASS CATEGORY'])
    ], remainder='passthrough')


In [13]:
preprocessor


,transformers,"[('cod_taxPresent', ...), ('cod_taxTimeOfSale', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,"[['1', '1A', ...]]"
,dtype,'int'
,handle_unknown,'error'


In [14]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings("ignore")


model = Pipeline([

    # Preprocesamiento
    ('preprocessor', preprocessor),
    # Modelo
    ('regressor', DecisionTreeRegressor(random_state=1))
])


grid = {
    'regressor__ccp_alpha':np.logspace(-3, 3),
    'regressor__max_depth': [3, 5, 7, 10, None],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4]
    }


grid_search = GridSearchCV(
    estimator=model,
    param_grid=grid,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    cv=5
    )

grid_search.fit(X_train, y_train)

print(f'Best RMSE: {-grid_search.best_score_:.2f} with {grid_search.best_params_}')
print(f'Train RMSE: {-grid_search.score(X_train, y_train):.2f}')
print(f'Test RMSE: {-grid_search.score(X_test, y_test):.2f}')

Best RMSE: 415864.92 with {'regressor__ccp_alpha': np.float64(0.001), 'regressor__max_depth': 7, 'regressor__min_samples_leaf': 4, 'regressor__min_samples_split': 2}
Train RMSE: 377473.42
Test RMSE: 415945.29


In [15]:
importance_features = pd.DataFrame({
    'Feature': grid_search.best_estimator_.named_steps['preprocessor'].get_feature_names_out(),
    'Importance': grid_search.best_estimator_.named_steps['regressor'].feature_importances_
})

importance_features.sort_values('Importance', ascending=False)

,Feature,Importance
6,cod_teCardinalidad__NEIGHBORHOOD,0.542782
14,remainder__GROSS SQUARE FEET,0.232834
9,cod_teCardinalidad__BUILDING CLASS AT TIME OF ...,0.055559
8,cod_teCardinalidad__ZIP CODE,0.036633
13,remainder__LAND SQUARE FEET,0.027131
15,remainder__YEAR BUILT,0.026636
10,cod_teCardinalidad__BUILDING CLASS CATEGORY,0.022492
7,cod_teCardinalidad__BUILDING CLASS AT PRESENT,0.016217
11,remainder__RESIDENTIAL UNITS,0.011018
3,cod_oh__BOROUGH_3,0.009966


## Modelo de boosting

In [16]:
from xgboost import XGBRegressor

boosting_model = Pipeline([

    # Preprocesamiento
    ('preprocessor', preprocessor),

    # Modelo
    ('regressor', XGBRegressor(
        random_state=1,
        objective='reg:squarederror'
    ))
])

boosting_grid = {

    'regressor__n_estimators': [100, 200],

    'regressor__max_depth': [3, 5, 7],

    'regressor__learning_rate': [0.01, 0.1],

    'regressor__subsample': [0.8, 1],

    'regressor__colsample_bytree': [0.8, 1]
}

boosting_search = GridSearchCV(
    estimator=boosting_model,
    param_grid=boosting_grid,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    cv=5
)

boosting_search.fit(X_train, y_train)

print(f'Best RMSE: {-boosting_search.best_score_:.2f} with {boosting_search.best_params_}')
print(f'Train RMSE: {-boosting_search.score(X_train, y_train):.2f}')
print(f'Test RMSE: {-boosting_search.score(X_test, y_test):.2f}')

Best RMSE: 372708.29 with {'regressor__colsample_bytree': 0.8, 'regressor__learning_rate': 0.1, 'regressor__max_depth': 7, 'regressor__n_estimators': 200, 'regressor__subsample': 1}
Train RMSE: 272314.02
Test RMSE: 367970.01


In [17]:
importance_features_boosting = pd.DataFrame({
    'Feature': boosting_search.best_estimator_.named_steps['preprocessor'].get_feature_names_out(),
    'Importance': boosting_search.best_estimator_.named_steps['regressor'].feature_importances_
})

importance_features_boosting.sort_values('Importance',ascending=False)

,Feature,Importance
6,cod_teCardinalidad__NEIGHBORHOOD,0.163395
14,remainder__GROSS SQUARE FEET,0.128969
9,cod_teCardinalidad__BUILDING CLASS AT TIME OF ...,0.082071
3,cod_oh__BOROUGH_3,0.069835
8,cod_teCardinalidad__ZIP CODE,0.063940
2,cod_oh__BOROUGH_2,0.056325
7,cod_teCardinalidad__BUILDING CLASS AT PRESENT,0.055800
11,remainder__RESIDENTIAL UNITS,0.046771
4,cod_oh__BOROUGH_4,0.042366
13,remainder__LAND SQUARE FEET,0.041756


## Importancia de características e interpretación de variables

A continuación, se presentan las variables más relevantes identificadas por cada modelo junto con una interpretación de su impacto sobre la predicción del precio de venta de las propiedades.

---

## RidgeCV (Modelo lineal)

### Variables más importantes

| Feature | Coeficiente |
|---|---|
| num_prep__GROSS SQUARE FEET | 187016.91 |
| num_prep2__RESIDENTIAL UNITS | -111886.96 |
| num_prep__LAND SQUARE FEET | 73574.49 |
| cod_oh__BOROUGH_4 | -69418.15 |
| cod_oh__BOROUGH_5 | -62186.70 |
| cod_taxTimeOfSale__TAX CLASS AT TIME OF SALE | -51307.65 |
| cod_oh__BOROUGH_3 | 37180.48 |
| num_prep2__SALE YEAR | 29552.60 |
| cod_taxPresent__TAX CLASS AT PRESENT | -17374.18 |
| num_prep2__SALE MONTH | 16466.87 |

### Interpretación

En el modelo lineal, la variable con mayor impacto positivo fue **GROSS SQUARE FEET**, lo que indica que propiedades con mayor área construida tienden a tener precios de venta más altos. De igual manera, **LAND SQUARE FEET** también mostró una relación positiva importante con el precio.

Por otro lado, **RESIDENTIAL UNITS** presentó un coeficiente negativo considerable, lo que podría indicar que ciertas propiedades con múltiples unidades residenciales están asociadas a precios promedio menores dentro del dataset.

Las variables relacionadas con la ubicación (**BOROUGH**) también tuvieron un papel importante en el modelo. Algunos distritos mostraron coeficientes negativos, lo cual sugiere diferencias significativas en el valor promedio de las propiedades dependiendo de la zona geográfica.

Además, variables temporales como **SALE YEAR** y **SALE MONTH** tuvieron influencia sobre el precio, indicando posibles cambios en el mercado inmobiliario a través del tiempo.

---

## Decision Tree Regressor

### Variables más importantes

| Feature | Importance |
|---|---|
| cod_teCardinalidad__NEIGHBORHOOD | 0.542782 |
| remainder__GROSS SQUARE FEET | 0.232834 |
| cod_teCardinalidad__BUILDING CLASS AT TIME OF SALE | 0.055559 |
| cod_teCardinalidad__ZIP CODE | 0.036633 |
| remainder__LAND SQUARE FEET | 0.027131 |
| remainder__YEAR BUILT | 0.026636 |
| cod_teCardinalidad__BUILDING CLASS CATEGORY | 0.022492 |
| cod_teCardinalidad__BUILDING CLASS AT PRESENT | 0.016217 |
| remainder__RESIDENTIAL UNITS | 0.011018 |
| cod_oh__BOROUGH_3 | 0.009966 |

### Interpretación

En el árbol de decisión, la variable más importante fue **NEIGHBORHOOD**, representando más del 54% de la importancia total del modelo. Esto indica que la ubicación específica de la propiedad es el factor más determinante para estimar su precio de venta.

La segunda variable más relevante fue **GROSS SQUARE FEET**, confirmando que el tamaño de la propiedad influye fuertemente en el valor final.

También destacaron variables relacionadas con la clasificación de edificios y ubicación geográfica como **BUILDING CLASS AT TIME OF SALE**, **ZIP CODE** y **BOROUGH**, lo cual evidencia que las características urbanas y el tipo de inmueble tienen un impacto importante en el precio.

Algunas variables temporales como **SALE YEAR** y **SALE MONTH** tuvieron poca o ninguna relevancia dentro del árbol, indicando que el modelo encontró patrones más fuertes en las características físicas y de ubicación de las propiedades.

---

## XGBoost Regressor

### Variables más importantes

| Feature | Importance |
|---|---|
| cod_teCardinalidad__NEIGHBORHOOD | 0.163395 |
| remainder__GROSS SQUARE FEET | 0.128969 |
| cod_teCardinalidad__BUILDING CLASS AT TIME OF SALE | 0.082071 |
| cod_oh__BOROUGH_3 | 0.069835 |
| cod_teCardinalidad__ZIP CODE | 0.063940 |
| cod_oh__BOROUGH_2 | 0.056325 |
| cod_teCardinalidad__BUILDING CLASS AT PRESENT | 0.055800 |
| remainder__RESIDENTIAL UNITS | 0.046771 |
| cod_oh__BOROUGH_4 | 0.042366 |
| remainder__LAND SQUARE FEET | 0.041756 |

### Interpretación

El modelo XGBoost mostró una distribución más equilibrada de la importancia entre las variables. Aunque **NEIGHBORHOOD** continuó siendo la característica más relevante, su importancia fue menor en comparación con el árbol de decisión, indicando que el modelo aprovecha información de múltiples variables simultáneamente.

Las variables relacionadas con ubicación (**NEIGHBORHOOD**, **ZIP CODE** y **BOROUGH**) siguieron siendo fundamentales, reafirmando que la localización es uno de los factores más influyentes en el precio de las propiedades.

Asimismo, variables físicas como **GROSS SQUARE FEET**, **LAND SQUARE FEET** y **RESIDENTIAL UNITS** tuvieron una contribución importante, demostrando que el tamaño y capacidad del inmueble afectan significativamente el valor de venta.

Las variables relacionadas con la clasificación del edificio también presentaron una importancia considerable, lo que indica que el tipo de propiedad y su uso influyen directamente en la estimación del precio.

---

## Conclusión general sobre las variables importantes

Los tres modelos coincidieron en que las variables relacionadas con la ubicación y el tamaño de las propiedades son los factores más relevantes para predecir el precio de venta.

Particularmente, **NEIGHBORHOOD**, **GROSS SQUARE FEET**, **LAND SQUARE FEET** y las variables asociadas al tipo de edificio aparecieron repetidamente entre las características más importantes. Esto sugiere que el valor de las propiedades en el dataset está fuertemente determinado por factores geográficos y físicos.

Además, los modelos basados en árboles lograron capturar de mejor manera relaciones complejas entre estas variables, especialmente XGBoost, lo cual explica su mejor desempeño predictivo frente al modelo lineal.

## Comparación de modelos y análisis de resultados

Para garantizar una comparación justa entre los modelos, se utilizaron las mismas particiones de entrenamiento y prueba en todos los experimentos. Además, la sintonización de hiperparámetros se realizó mediante validación cruzada de 5 particiones (*cv=5*) utilizando la métrica RMSE (*neg_root_mean_squared_error*). También se mantuvo un *random_state=1* en los modelos donde correspondía, asegurando la reproducibilidad de los resultados.

### Tabla comparativa de resultados

| Modelo | Mejor RMSE CV | Train RMSE | Test RMSE | Observaciones |
|---|---|---|---|---|
| RidgeCV (Lineal) | 421057.63 | 407635.92 | 420376.46 | Modelo más simple y con menor sobreajuste |
| Decision Tree Regressor | 415864.92 | 377473.42 | 415945.29 | Mejora moderada frente al modelo lineal |
| XGBoost Regressor | 372708.29 | 272314.02 | 367970.01 | Mejor desempeño general |

### Análisis comparativo

El modelo lineal RidgeCV obtuvo el mayor error de prueba entre los tres modelos evaluados, con un RMSE de aproximadamente 420 mil. Aunque presentó un comportamiento estable entre entrenamiento y prueba, su capacidad para capturar relaciones complejas entre las variables fue limitada debido a la naturaleza lineal del modelo.

El árbol de decisión logró mejorar ligeramente el desempeño respecto al modelo lineal, reduciendo el RMSE de prueba a aproximadamente 416 mil. Esto indica que el modelo pudo capturar relaciones no lineales presentes en los datos. Sin embargo, la diferencia entre el RMSE de entrenamiento y prueba evidencia cierto nivel de sobreajuste, aunque todavía moderado gracias a la sintonización de hiperparámetros como *max_depth*, *min_samples_leaf* y *ccp_alpha*.

El modelo de boosting basado en XGBoost presentó el mejor rendimiento general. Obtuvo el menor RMSE tanto en validación cruzada como en el conjunto de prueba, alcanzando aproximadamente 368 mil en prueba. Esto demuestra una mayor capacidad para modelar patrones complejos y relaciones no lineales entre las variables. Sin embargo, la diferencia considerable entre el error de entrenamiento y prueba sugiere un mayor nivel de sobreajuste en comparación con los otros modelos, aunque el desempeño en prueba sigue siendo superior.

### Hiperparámetros óptimos encontrados

#### RidgeCV
- *alpha = 75.43120063354607*

#### Decision Tree Regressor
- *ccp_alpha = 0.001*
- *max_depth = 7*
- *min_samples_leaf = 4*
- *min_samples_split = 2*

#### XGBoost Regressor
- *n_estimators = 200*
- *max_depth = 7*
- *learning_rate = 0.1*
- *subsample = 1*
- *colsample_bytree = 0.8*

### Conclusión y recomendación

Con base en los resultados obtenidos, el modelo XGBoost es el más adecuado para este problema, ya que alcanzó el menor RMSE en validación y prueba, mostrando una capacidad superior para realizar predicciones más precisas. Aunque presenta señales de sobreajuste, su desempeño general sigue siendo significativamente mejor que el de los modelos lineal y árbol de decisión.

El modelo RidgeCV puede considerarse una buena alternativa cuando se prioriza interpretabilidad y simplicidad, mientras que el árbol de decisión representa una solución intermedia con capacidad para modelar relaciones no lineales, aunque con menor desempeño que el modelo de boosting.

### Evaluación general del desempeño sobre el dataset

Los resultados obtenidos indican que el dataset presenta relaciones no lineales importantes entre las variables predictoras y el precio de venta de las propiedades. Esto se evidencia en el mejor desempeño de los modelos basados en árboles frente al modelo lineal.

El modelo XGBoost logró capturar de manera más efectiva estos patrones complejos, obteniendo la menor tasa de error entre todos los modelos evaluados. Aunque el RMSE sigue siendo relativamente alto, esto puede explicarse por la gran variabilidad y dispersión presente en los precios de las propiedades dentro del dataset de viviendas de Nueva York, donde existen diferencias significativas entre zonas, tamaños y tipos de inmuebles.

En general, puede concluirse que los modelos implementados sí logran aprender patrones útiles del dataset y generar predicciones razonables, siendo XGBoost el modelo más adecuado para este problema debido a su mejor capacidad predictiva y desempeño general.